In [1]:
%%time 
%pip install transformers~=4.40.0
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu torchtext==0.18.0+cpu  \
    --index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
ERROR: Could not find a version that satisfies the requirement torch==2.8.0+cpu (from versions: 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.8.0+cpu
Note: you may need to restart the kernel to use updated packages.
CPU times: user 36.8 ms, sys: 27.8 ms, total: 64.6 ms
Wall time: 3.03 s


In [2]:
from transformers import pipeline
from transformers import DistilBertTokenizer,DistilBertForSequenceClassification
import torch
from transformers import GPT2LMHeadModel,GPT2Tokenizer
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [3]:
# Load the tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

In [4]:
# Sample text
text = "Congratulations! You've won a free ticket to the Bahamas. Reply WIN to claim."
# Tokenize the input text
inputs = tokenizer(text,return_tensors = "pt")

print(inputs)

{'input_ids': tensor([[  101, 23156,   999,  2017,  1005,  2310,  2180,  1037,  2489,  7281,
          2000,  1996, 17094,  1012,  7514,  2663,  2000,  4366,  1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [5]:
# Perform inference
with torch.no_grad():
    outputs = model(**inputs)
print(outputs)    

SequenceClassifierOutput(loss=None, logits=tensor([[-3.9954,  4.3336]]), hidden_states=None, attentions=None)


In [6]:
logits = outputs.logits
logits.shape

torch.Size([1, 2])

In [7]:
# Convert logits to probabilities
probs = torch.softmax(logits , dim = -1)

# Get the predicted class
predicted_class = torch.argmax(probs, dim = -1)
# Map the predicted class to the label
labels = ["NEGATIVE", "POSITIVE"]
predicted_label = labels[predicted_class]

print(f"Predicted label is :{predicted_label}")

Predicted label is :POSITIVE


In [8]:
# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")


In [9]:
# Load the tokenizer and model
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [10]:
# Prompt
prompt = "Once upon a time"

inputs = tokenizer(prompt, return_tensors= "pt")
print(inputs)

{'input_ids': tensor([[7454, 2402,  257,  640]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


In [11]:
# Generate text
outputs_id = model.generate(
    inputs = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    pad_token_id = tokenizer.eos_token_id,
    max_length = 50,
    num_return_sequences = 1
)
outputs_id

tensor([[7454, 2402,  257,  640,   11,  262,  995,  373,  257, 1295,  286, 1049,
         8737,  290, 1049, 3514,   13,  383,  995,  373,  257, 1295,  286, 1049,
         3514,   11,  290,  262,  995,  373,  257, 1295,  286, 1049, 3514,   13,
          383,  995,  373,  257, 1295,  286, 1049, 3514,   11,  290,  262,  995,
          373,  257]])

In [12]:
# Decode the generated text
generated_text = tokenizer.decode(outputs_id[0],skip_special_tokens = True)

print(generated_text)

Once upon a time, the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a


In [14]:
# Load a general text classification model
classifier = pipeline("text-classification", model = "distilbert-base-uncased-finetuned-sst-2-english")
# Classify a sample text
result = classifier("Congratulations! You've won a free ticket to the Bahamas. Reply WIN to claim.")
print(result)

[{'label': 'POSITIVE', 'score': 0.9997586607933044}]


In [15]:
#Language detection using pipeline
classifier = pipeline("text-classification", model ="papluca/xlm-roberta-base-language-detection")
result = classifier("Bonjour, comment ça va?")
print(result)

config.json: 0.00B [00:00, ?B/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'fr', 'score': 0.9934879541397095}]


In [16]:
# Initialize the text generation pipeline with GPT-2
generator = pipeline("text-generation", model="gpt2")
# Generate text based on a given prompt
prompt = "Once upon a time"
result = generator(prompt, max_length=50, num_return_sequences=1, truncation=True)

# Print the generated text
print(result[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time she got a bit nervous because there are no other females in the school, but she was not looking at the girls, which got her nervous for a while. I guess it was her fault, but it's fine because she doesn


In [17]:
# Initialize the text generation pipeline with T5
generator = pipeline("text2text-generation", model="t5-small")
#Generate text based on a given prompt
prompt = "translate English to French: How are you?"
result = generator(prompt, max_length=50, num_return_sequences=1)

# Print the generated text
print(result[0]['generated_text'])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Comment êtes-vous?


In [18]:
#Exercise: Fill-mask task using BERT with pipeline()
# Initialize the fill-mask pipeline with BERT
fill_mask = pipeline("fill-mask", model = "bert-base-uncased")
# Generate text by filling in the masked token
prompt = "The capital of France is [MASK]."
result = fill_mask(prompt)
# Print the generated text
print(result)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[{'score': 0.41679075360298157, 'token': 3000, 'token_str': 'paris', 'sequence': 'the capital of france is paris.'}, {'score': 0.07141678035259247, 'token': 22479, 'token_str': 'lille', 'sequence': 'the capital of france is lille.'}, {'score': 0.06339383125305176, 'token': 10241, 'token_str': 'lyon', 'sequence': 'the capital of france is lyon.'}, {'score': 0.044448137283325195, 'token': 16766, 'token_str': 'marseille', 'sequence': 'the capital of france is marseille.'}, {'score': 0.030297210440039635, 'token': 7562, 'token_str': 'tours', 'sequence': 'the capital of france is tours.'}]
